## 1. 투시 보정하기: 필요없어서 안하기로함 괜히 이미지수만늘어나는듯..

## 2. Grid split 수행 코드

🧐 코드 핵심 포인트 (박사님 확인용)
똑똑한 복사 (Step 2): 1.3만 장을 다 옮기지 않습니다. CSV에서 선발된 877장만 찾아서 로컬로 가져옵니다. 드라이브 부하를 최소화하는 핵심 로직입니다.

격자 분할 (Step 3): 주신 노트북의 12분할 개념을 적용했습니다. TARGET_LETTUCE_INDICES에 [2, 4, 9]를 넣어서 딱 필요한 상추만 뽑아냅니다.

데이터 절감: 13,000장 대신 877장을 처리하므로 속도가 매우 빠를 겁니다. 결과물은 약 2,631장이 생성됩니다.

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# ========================= [USER SETTINGS] ========================= #
# 1. 원본 CSV 파일 경로
CSV_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/260102_pixel_scale_final.csv"

# 2. 원본 베드 ROI 이미지가 들어있는 드라이브 폴더 (1.3만 장 있는 곳)
DRIVE_INPUT_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/2. ROI_BOX/251213-251226"

# 3. 최종 상추 크롭 이미지를 저장할 드라이브 폴더
DRIVE_OUTPUT_ROOT = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251213-251226"

# 4. 상추 번호 선택 (2, 4, 9번)
TARGET_LETTUCE_INDICES = [2, 4, 9]

NUM_WORKERS = 4
# =================================================================== #

In [ ]:


# 로컬 작업 경로 (수정 불필요)
LOCAL_SELECTED_DIR = "/content/selected_beds"
LOCAL_CROP_DIR = "/content/lettuce_results"

def step1_select_images():
    print("Step 1: CSV 분석 및 이미지 선별 중...")
    df = pd.read_csv(CSV_PATH)

    # 시간 추출 및 정렬 (파일명 포맷: bedXX_YYYYMMDD_HHMMSS_...)
    df['time'] = df['image_name'].apply(lambda x: x.split('_')[2] if len(x.split('_')) > 2 else "999999")
    df_sorted = df.sort_values(by=['bed_date', 'time'])

    # 베드_날짜별 최소 시간 1장 선별
    df_selected = df_sorted.groupby('bed_date').first().reset_index()

    # 선택된 리스트 CSV 저장
    df_selected.to_csv('/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226/260103_pixel_scale_selected.csv', index=False)
    print(f"✅ 선별 완료: {len(df_selected)}장의 베드 이미지 선정.")
    return df_selected['image_name'].tolist()

def step2_copy_to_local(selected_names):
    print("Step 2: 선별된 이미지만 로컬(/content/)로 복사 중...")
    os.makedirs(LOCAL_SELECTED_DIR, exist_ok=True)

    # 드라이브 전체 탐색하여 파일 찾기 (하위 폴더 대응)
    found_count = 0
    selected_set = set(selected_names)

    for root, _, files in os.walk(DRIVE_INPUT_DIR):
        for f in files:
            if f in selected_set:
                shutil.copy(os.path.join(root, f), os.path.join(LOCAL_SELECTED_DIR, f))
                found_count += 1

    print(f"✅ 복사 완료: {found_count}장 복사됨.")

def step3_crop_logic():
    print("Step 3: 상추 단위 2차 Crop 시작 (베드별 폴더 생성)...")
    os.makedirs(LOCAL_CROP_DIR, exist_ok=True)

    image_paths = list(Path(LOCAL_SELECTED_DIR).glob("*.*"))

    def process_single_bed(img_path):
        img = cv2.imread(str(img_path))
        if img is None: return
        h, w = img.shape[:2]

        # 1. 베드명으로 하위 폴더 경로 설정 (예: bed00_20251213)
        bed_folder_name = img_path.stem
        bed_out_dir = Path(LOCAL_CROP_DIR) / bed_folder_name
        bed_out_dir.mkdir(parents=True, exist_ok=True) # 폴더 생성

        rows, cols = 3, 4
        unit_h, unit_w = h // rows, w // cols

        count = 1
        for r in range(rows):
            for c in range(cols):
                if count in TARGET_LETTUCE_INDICES:
                    y1, y2 = r * unit_h, (r + 1) * unit_h
                    x1, x2 = c * unit_w, (c + 1) * unit_w
                    lettuce_img = img[y1:y2, x1:x2]

                    # 2. 하위 폴더 안에 저장 (예: lettuce_02.jpg)
                    out_name = f"lettuce_{count:02d}.jpg"
                    cv2.imwrite(str(bed_out_dir / out_name), lettuce_img)
                count += 1

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        ex.map(process_single_bed, image_paths)
    print("✅ 크롭 완료.")

def step4_upload_and_clean():
    print("Step 4: 결과 전송 및 정리 중...")
    # 결과물 압축
    output_zip = "/content/lettuce_final.zip"
    os.system(f"zip -r -q {output_zip} {LOCAL_CROP_DIR}")

    # 드라이브로 복사
    os.makedirs(DRIVE_OUTPUT_ROOT, exist_ok=True)
    shutil.copy(output_zip, os.path.join(DRIVE_OUTPUT_ROOT, "lettuce_final.zip"))

    # 정리
    shutil.rmtree(LOCAL_SELECTED_DIR)
    shutil.rmtree(LOCAL_CROP_DIR)
    if os.path.exists(output_zip): os.remove(output_zip)
    print(f"🎉 모든 작업 종료! 드라이브 확인: {DRIVE_OUTPUT_ROOT}")

if __name__ == "__main__":
    selected_list = step1_select_images()
    step2_copy_to_local(selected_list)
    step3_crop_logic()
    step4_upload_and_clean()

Step 1: CSV 분석 및 이미지 선별 중...
✅ 선별 완료: 877장의 베드 이미지 선정.
Step 2: 선별된 이미지만 로컬(/content/)로 복사 중...
✅ 복사 완료: 877장 복사됨.
Step 3: 상추 단위 2차 Crop 시작 (베드별 폴더 생성)...
✅ 크롭 완료.
Step 4: 결과 전송 및 정리 중...
🎉 모든 작업 종료! 드라이브 확인: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251213-251226
